# 01 — Data Cleaning

The synthetic dataset intentionally includes real-world messiness:
- Inconsistent casing/whitespace in `source_channel` (~8% of rows)
- Missing values in `years_experience` (~5%) and `expected_salary` (~4%)
- Duplicate candidate rows (~1.5%)

This notebook loads the raw data, documents every cleaning decision, and writes a clean version back to the database for use in downstream notebooks.

In [1]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = "../database/recruitment.db"
conn = sqlite3.connect(DB_PATH)

candidates    = pd.read_sql("SELECT * FROM candidates",    conn)
pipeline      = pd.read_sql("SELECT * FROM pipeline_stages", conn)
roles         = pd.read_sql("SELECT * FROM roles",         conn)
clients       = pd.read_sql("SELECT * FROM clients",       conn)

print(f"candidates: {len(candidates):,} rows")
print(f"pipeline_stages: {len(pipeline):,} rows")
print(f"roles: {len(roles):,} rows")
print(f"clients: {len(clients):,} rows")

candidates: 1,015 rows
pipeline_stages: 3,041 rows
roles: 50 rows
clients: 18 rows


## 1. Inspect raw data

In [2]:
candidates.head(10)

,candidate_id,source_channel,role_applied,years_experience,expected_salary,date_sourced
0,1,Job Board,Marketing Coordinator,6.0,66000.0,2024-09-27
1,2,LinkedIn,Software Engineer,4.0,73000.0,2025-03-28
2,3,Cold Outreach,Data Analyst,13.0,79000.0,2024-02-10
3,4,Job Board,Accountant,13.0,59000.0,2024-08-30
4,5,Referral,Accountant,10.0,92000.0,2024-09-11
5,6,Agency Database,Accountant,15.0,98000.0,2024-08-21
6,7,Job Board,Software Engineer,2.0,75000.0,2025-06-05
7,8,Job Board,Marketing Coordinator,NaN,95000.0,2024-11-17
8,9,Cold Outreach,Software Engineer,6.0,48000.0,2024-12-04
9,10,Cold Outreach,Data Analyst,13.0,77000.0,2024-07-15


In [3]:
candidates.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   candidate_id      1015 non-null   int64  
 1   source_channel    1015 non-null   object 
 2   role_applied      1015 non-null   object 
 3   years_experience  974 non-null    float64
 4   expected_salary   971 non-null    float64
 5   date_sourced      1015 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 47.7+ KB


In [4]:
# Missing value summary
missing = candidates.isnull().sum()
missing_pct = (missing / len(candidates) * 100).round(1)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
candidate_id,0,0.0
source_channel,0,0.0
role_applied,0,0.0
years_experience,41,4.0
expected_salary,44,4.3
date_sourced,0,0.0


## 2. Fix `source_channel` — inconsistent casing and whitespace

In [5]:
# Show the raw variants before cleaning
print("Raw source_channel values:")
print(candidates['source_channel'].value_counts(dropna=False).to_string())

Raw source_channel values:
source_channel
Job Board            206
Agency Database      200
Cold Outreach        176
LinkedIn             168
Referral             166
cold outreach         11
LINKEDIN              10
AGENCY DATABASE       10
linkedin               9
referral               8
REFERRAL               8
job board              7
 LinkedIn              7
COLD OUTREACH          7
 Agency Database       5
 Job Board             4
JOB BOARD              4
agency database        4
 Cold Outreach         3
 Referral              2


In [6]:
CANONICAL_CHANNELS = {
    'linkedin':         'LinkedIn',
    'referral':         'Referral',
    'job board':        'Job Board',
    'agency database':  'Agency Database',
    'cold outreach':    'Cold Outreach',
}

def normalise_channel(raw):
    import pandas as _pd
    if _pd.isna(raw):
        return raw
    return CANONICAL_CHANNELS.get(raw.strip().lower(), raw.strip())

# Count messy rows BEFORE in-place modification
n_channel_issues = int(
    candidates['source_channel'].apply(
        lambda v: not pd.isna(v) and v != normalise_channel(v)
    ).sum()
)

candidates['source_channel'] = candidates['source_channel'].apply(normalise_channel)

print('After normalisation:')
print(candidates['source_channel'].value_counts(dropna=False).to_string())


After normalisation:
source_channel
Job Board          221
Agency Database    219
Cold Outreach      197
LinkedIn           194
Referral           184


## 3. Handle missing values

In [7]:
# years_experience: missing ~5%
# Decision: impute with median. Years of experience is right-skewed;
# median is more robust than mean. Flag imputed rows.
exp_median = candidates['years_experience'].median()
candidates['exp_imputed'] = candidates['years_experience'].isna()
candidates['years_experience'] = candidates['years_experience'].fillna(exp_median)

print(f"Median years_experience used for imputation: {exp_median}")
print(f"Rows imputed: {candidates['exp_imputed'].sum()}")

Median years_experience used for imputation: 8.0
Rows imputed: 41


In [8]:
# expected_salary: missing ~4%
# Decision: impute with channel-level median salary.
# Salary expectations vary meaningfully by sourcing channel
# (e.g., LinkedIn candidates skew higher than Agency Database).
channel_salary_median = candidates.groupby('source_channel')['expected_salary'].median()
print("Median expected salary by channel:")
print(channel_salary_median.to_string())

def impute_salary(row):
    if pd.isna(row['expected_salary']):
        return channel_salary_median.get(row['source_channel'], candidates['expected_salary'].median())
    return row['expected_salary']

candidates['salary_imputed'] = candidates['expected_salary'].isna()
candidates['expected_salary'] = candidates.apply(impute_salary, axis=1)

print(f"\nRows imputed: {candidates['salary_imputed'].sum()}")

Median expected salary by channel:
source_channel
Agency Database    76000.0
Cold Outreach      77000.0
Job Board          71000.0
LinkedIn           75000.0
Referral           78000.0

Rows imputed: 44


## 4. Identify and remove duplicate candidates

In [9]:
# Duplicates were injected as identical rows (same fields, new candidate_id).
# Detection: group by non-ID fields and flag rows where all content columns match.
content_cols = ['source_channel', 'role_applied', 'years_experience',
                'expected_salary', 'date_sourced']

dupes_mask = candidates.duplicated(subset=content_cols, keep='first')
print(f"Duplicate rows found: {dupes_mask.sum()}")
print(f"({dupes_mask.sum() / len(candidates) * 100:.1f}% of total)")

Duplicate rows found: 15
(1.5% of total)


In [10]:
# Decision: keep the first occurrence (lowest candidate_id), drop the rest.
# In a real system you'd reconcile pipeline history for both IDs first;
# here the duplicate rows have no pipeline records, so a simple drop is safe.
candidates_clean = candidates[~dupes_mask].copy()
print(f"Rows before deduplication: {len(candidates):,}")
print(f"Rows after  deduplication: {len(candidates_clean):,}")

Rows before deduplication: 1,015
Rows after  deduplication: 1,000


## 5. Parse date columns

In [11]:
candidates_clean['date_sourced'] = pd.to_datetime(candidates_clean['date_sourced'])
pipeline['stage_date']           = pd.to_datetime(pipeline['stage_date'])
roles['date_opened']             = pd.to_datetime(roles['date_opened'])
roles['date_closed']             = pd.to_datetime(roles['date_closed'])

print("Date range (candidates sourced):")
print(f"  {candidates_clean['date_sourced'].min().date()} → {candidates_clean['date_sourced'].max().date()}")

Date range (candidates sourced):
  2024-01-01 → 2025-06-20


## 6. Cleaning summary

In [12]:
summary = __import__('pandas').DataFrame([
    {'issue': 'Inconsistent source_channel casing/whitespace',
     'rows_affected': n_channel_issues,
     'action': 'strip().lower() -> canonical title-case mapping'},
    {'issue': 'Missing years_experience',
     'rows_affected': int(candidates['exp_imputed'].sum()),
     'action': f'Imputed with overall median ({exp_median})'},
    {'issue': 'Missing expected_salary',
     'rows_affected': int(candidates['salary_imputed'].sum()),
     'action': 'Imputed with per-channel median salary'},
    {'issue': 'Duplicate candidate rows',
     'rows_affected': int(dupes_mask.sum()),
     'action': 'Kept first occurrence by candidate_id, dropped rest'},
])
summary


,issue,rows_affected,action
0,Inconsistent source_channel casing/whitespace,99,strip().lower() -> canonical title-case mapping
1,Missing years_experience,41,Imputed with overall median (8.0)
2,Missing expected_salary,44,Imputed with per-channel median salary
3,Duplicate candidate rows,15,"Kept first occurrence by candidate_id, dropped..."


In [13]:
# Persist cleaned candidates to a new table for downstream notebooks.
# This keeps raw data untouched and makes the cleaning step explicit.
candidates_clean[['candidate_id', 'source_channel', 'role_applied',
                   'years_experience', 'expected_salary', 'date_sourced']].to_sql(
    'candidates_clean', conn, if_exists='replace', index=False
)
conn.commit()
print("candidates_clean written to database.")
conn.close()

candidates_clean written to database.
